# 01 — Hosted Lightning and Ultra Text2SQL baselines

This CPU-only notebook evaluates NVIDIA-hosted Nemotron 3.5 Lightning and Nemotron 3 Ultra on the same frozen BIRD Mini-Dev SQLite questions. Each model sees database DDL, the natural-language question, and BIRD's evidence, then must produce SQL.

The primary score is **execution accuracy**: predicted and reference SQL must return the same result on the official database. SQL validity, executability, and normalized string match are diagnostics. The default is 25 requests per model—50 total—not hundreds of calls. Responses checkpoint after every request and resume safely after a 429 or interruption.

Hosted models are useful task targets, but they are NVFP4 services. Notebook 02 repeats the full 100-row holdout on the exact local BF16 checkpoint that Notebook 03 tunes.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, time

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
ARTIFACTS_DIR = Path(os.environ.get('NEMOTRON_ARTIFACTS_DIR', ROOT / 'artifacts')).expanduser().resolve()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = ARTIFACTS_DIR / 'data/bird-text2sql'
print('Repository:', ROOT)
print('Artifacts:', ARTIFACTS_DIR)


## 1. API preflight and public/private endpoint switch

Copy `config/api.local.toml.example` to the ignored `config/api.local.toml` for an internal OpenAI-compatible endpoint. Set `API_PROFILE_OVERRIDE` to `'public'` or `'local'` and rerun this cell plus authentication; leave it `None` for automatic selection. Keys remain in the environment or notebook input, never in TOML.


In [ ]:
subprocess.run([sys.executable, 'scripts/preflight.py', '--profile', 'api'], check=True)

from nemotron_ft_lab.api_config import load_nvidia_api_config

API_PROFILE_OVERRIDE = None  # None=auto, or use 'public' / 'local'
API_CONFIG = load_nvidia_api_config(ROOT, profile=API_PROFILE_OVERRIDE)
print('API profile:', API_CONFIG.profile_name, f'({API_CONFIG.source_label})')
print('Endpoint:', API_CONFIG.base_url)
print('Lightning:', API_CONFIG.lightning.model_id, '->', API_CONFIG.lightning.served_variant)
print('Ultra:', API_CONFIG.ultra.model_id, '->', API_CONFIG.ultra.served_variant)
print('Rate limit used by this notebook:', API_CONFIG.requests_per_minute, 'requests/minute')


## 2. Freeze the executable evaluation bundle

The first run downloads the official BIRD Mini-Dev package (~800 MB compressed), extracts its SQLite databases, and selects 100 rows before any model is scored. Selection preserves the benchmark's difficulty mix and spreads rows across all 11 databases. The manifest records a pinned exclusion for question 701, whose official gold query exceeded the 30-second audit timeout; preparation fails instead of silently choosing different IDs on slower hardware. Training later uses only BIRD train mirrors; Mini-Dev is never training data.


In [ ]:
subprocess.run([
    sys.executable, 'scripts/prepare_text2sql.py',
    '--output-dir', str(DATA_DIR), '--evaluation-only',
], check=True)

from nemotron_ft_lab.constants import DEFAULT_API_EVALUATION_SIZE, DEFAULT_SEED
from nemotron_ft_lab.data import balanced_evaluation_subset, build_messages, read_jsonl

all_eval_rows = read_jsonl(DATA_DIR / 'evaluation.jsonl')
CLOUD_EVAL_SIZE = DEFAULT_API_EVALUATION_SIZE  # 25 requests/model; set <=100 before first run
cloud_rows = balanced_evaluation_subset(
    all_eval_rows, size=CLOUD_EVAL_SIZE, seed=DEFAULT_SEED + 1,
)
manifest = json.loads((DATA_DIR / 'evaluation_manifest.json').read_text())
print('Frozen local holdout:', len(all_eval_rows))
print('Cloud rows per model:', len(cloud_rows))
print('Cloud requests across Lightning + Ultra:', 2 * len(cloud_rows))
print('Difficulty mix:', manifest['difficulty_distribution'])
print('Databases:', len(manifest['database_distribution']))
print('Pinned gold-query exclusions:', manifest['excluded_gold_execution_question_ids'])
print()
print('Example user message:')
print(build_messages(cloud_rows[0])[1]['content'][:1800])


## 3. Authenticate without storing the key

Prefer exporting `NVIDIA_API_KEY` before starting Jupyter. Otherwise the built-in `input()` prompt is used because some notebook password widgets cannot accept paste. Input is briefly visible, then cleared. The key is never written to an artifact.


In [ ]:
from openai import OpenAI
from IPython.display import clear_output

api_key = os.environ.get('NVIDIA_API_KEY', '').strip()
used_native_prompt = not api_key
if used_native_prompt:
    api_key = input('Paste NVIDIA API key (visible until Enter), then press Enter: ').strip()
    clear_output(wait=False)
if not api_key:
    raise RuntimeError('An NVIDIA API key is required for hosted baselines.')
client = OpenAI(
    base_url=API_CONFIG.base_url, api_key=api_key,
    max_retries=0, timeout=API_CONFIG.timeout_seconds,
)
del api_key
print('API client configured; input cleared.' if used_native_prompt else 'API client configured from NVIDIA_API_KEY.')


## 4. Run both hosted models and execute their SQL

Prompt protocol `v1` is identical for both models: empty system turn, then `schema`, blank line, `question`, blank line, optional `evidence`, with thinking disabled. Cache filenames include the API profile and sample size so internal/public runs cannot collide.


In [ ]:
from nemotron_ft_lab.evaluation import (
    generate_nvidia_api_predictions, paired_execution_comparison,
    save_report, score_predictions,
)

CLOUD_MODELS = {
    'lightning': API_CONFIG.lightning,
    'ultra': API_CONFIG.ultra,
}
api_reports = {}
for name, spec in CLOUD_MODELS.items():
    endpoint_model_fingerprint = hashlib.sha256(
        f'{API_CONFIG.base_url}\0{spec.model_id}\0{manifest["evaluation_sha256"]}\0v1'.encode()
    ).hexdigest()[:12]
    artifact_name = f'{API_CONFIG.artifact_prefix}{name}_{endpoint_model_fingerprint}'
    resume_path = ARTIFACTS_DIR / f'evaluation/api_{artifact_name}_text2sql_v1_{CLOUD_EVAL_SIZE}.jsonl'
    print()
    print(f'Evaluating {name}: {spec.model_id}')
    started = time.perf_counter()
    generated = generate_nvidia_api_predictions(
        client, cloud_rows, model=spec.model_id, resume_path=resume_path,
        requests_per_minute=API_CONFIG.requests_per_minute,
    )
    report = score_predictions(generated, data_dir=DATA_DIR)
    report.update({
        'wall_time_seconds': time.perf_counter() - started,
        'endpoint': API_CONFIG.base_url,
        'api_profile': API_CONFIG.profile_name,
        'served_variant': spec.served_variant,
        'precision': 'NVFP4',
        'prompt_protocol': 'bird-schema-question-evidence-v1',
    })
    report_path = ARTIFACTS_DIR / f'evaluation/baseline_api_{artifact_name}_text2sql_{CLOUD_EVAL_SIZE}.json'
    save_report(report_path, report, model=spec.model_id, run_type=f'hosted-{name}-text2sql')
    api_reports[name] = report

metrics = ('n', 'execution_accuracy', 'sql_valid_rate', 'sql_executable_rate', 'normalized_exact_match')
{name: {key: report[key] for key in metrics} for name, report in api_reports.items()}


In [ ]:
ultra_vs_lightning = paired_execution_comparison(
    api_reports['lightning'], api_reports['ultra'],
)
ultra_vs_lightning.update({
    'lightning_execution_accuracy': api_reports['lightning']['execution_accuracy'],
    'ultra_execution_accuracy': api_reports['ultra']['execution_accuracy'],
    'interpretation': 'Ultra minus Lightning on identical hosted requests',
})
print(json.dumps(ultra_vs_lightning, indent=2))

for name, report in api_reports.items():
    print()
    print(f'{name.title()} examples:')
    for row in report['rows'][:3]:
        print()
        print('Q:', row['question'])
        print('Gold:', row['expected_sql'])
        print('Generated:', row['generated'])
        print('Execution correct:', row['execution_correct'])


## Result contract

These hosted scores are task targets, not the causal fine-tuning comparison. Notebook 02 evaluates the pinned local BF16 base on all 100 frozen IDs. Notebook 03 evaluates the merged LoRA checkpoint on those same 100 IDs and bootstraps the paired execution-accuracy delta; it also compares tuned Lightning with hosted Lightning/Ultra only on their shared 25 IDs.
